In [1]:
import sympy as sym
import re

from IPython.display import display, Latex

In [29]:
class Reaction:

    
    def __init__(self, reaction: str) -> None:

        self.direction: str
        self.k: sym.Symbol =  sym.Symbol("k")
        self.left, self.right = self._parse_reaction(reaction)

    @staticmethod
    def _parse_term(s):
        pat = re.compile(r"^\s*(?:(\d+)\s*)?(\S+)\s*$")
        m = pat.match(s)
        if not m:
            raise ValueError(s)
        coeff = int(m.group(1)) if m.group(1) else 1
        name = sym.Symbol(m.group(2))
        return coeff, name

    def _asign(self, reaction: str) -> tuple[list[str],list[str]]:

        sign_dict = {
            "<=>": "eq",
            "<->": "eq",
            "<>": "eq",
            "<-": "left",
            "->": "right",
            "=": "right",
            ">": "right",
            "<": "left",
        }

        for key in sign_dict.keys():
            if key in reaction:
                self.direction = sign_dict[key]
                return reaction.split(key)
        
        
    def _parse_reaction(self, reaction: str) -> tuple:
    
        left, right = self._asign(reaction)
        
        left = [self._parse_term(x.strip()) for x in left.split("+")]
        right = [self._parse_term(x.strip()) for x in right.split("+")]
        #print(left, right)
        return left, right

    def __str__(self):

        sign_dict = {
            "right": "->",
            "left": "<-",
            "eq": "<->",
        }
        
        reagents = " + ".join([f"{x[0] if x[0] != 1 else ""} {x[1]}" for x in self.reagents])
        products = " + ".join([f"{x[0] if x[0] != 1 else ""} {x[1]}" for x in self.products])
        
        
        return f"{reagents} {sign_dict[self.direction]} {products}"

    def __eq__(self, other: Reaction):

        return all([self.left == other.left, self.right == other.right])
        
    def to_latex(self, show_k = False):

        sign_dict = {
            "right": "->",
            "left": "<-",
            "eq": "<->",
        }
        
        reagents = " + ".join([f"{x[0] if x[0] != 1 else ""} {x[1]}" for x in self.reagents])
        products = " + ".join([f"{x[0] if x[0] != 1 else ""} {x[1]}" for x in self.products])

        const = f"[{self.k}]" if show_k else "" 
        latex = f"\\ce{{{reagents} {sign_dict[self.direction]}{const} {products}}}"

        return latex

    @property
    def substances(self):

        prod = {x[1] for x in self.products}
        reag = {x[1] for x in self.reagents}

        return prod | reag
    
    @property
    def products(self) -> list[tuple[int, str]]:

        right = self.right
        if not self.direction in {"right", "eq"}:
            right = self.left

        return right

    @property
    def reagents(self) -> list[tuple[int, str]]:
        
        left = self.left
        if self.direction == "left":
            left = self.right

        return left

    @property
    def dict(self):

        return {"products": self.prod, "reagents": self.products, "direction": self.direction, "k": self.k}

                
    
    
       
    

In [33]:
reactions = [
    "A + B -> C",
    "A + B -> C",
    "A + B <- C",
    "2 A + 4 B -> 5 C",
    "4A + 5B -> 6C",
    "5 A + 6B -> 7 C",
    "2 H2O2 + 4D3 <-> 543 P3"
]

for reaction in reactions:
    reaction = Reaction(reaction)
    print(reaction.substances)
    display(Latex(f"${reaction.to_latex()}$"))

{A, C, B}


<IPython.core.display.Latex object>

{A, C, B}


<IPython.core.display.Latex object>

{A, C, B}


...

<IPython.core.display.Latex object>

{A, C, B}


<IPython.core.display.Latex object>

{P3, D3, H2O2}


<IPython.core.display.Latex object>

In [34]:
class ReactionChain:

    def __init__(self, reaction_chain: list[str] | str):

        if isinstance(reaction_chain, str):
            reaction_chain = reaction_chain.splitlines()
        
        reactions = [Reaction(x) for x in reaction_chain]
        self.reactions = self.merge_equations(reactions)
        
    @staticmethod
    def merge_equations(reactions: list[Reaction]) -> list[Reaction]:

        
        
#        reactions = [r.dict for r in reactions]

        new_reactions = []
        for i, ri in enumerate(reactions):
            for j, rj in enumerate(reactions):
                if i <= j:
                    continue
                
                

                print(ri, f"--{ri==rj}--", rj) 

        return reactions
    
    def __str__(self):
        return "\n".join(str(x) for x in self.reactions)

    def to_latex(self, show_k: bool = True):
        latex = "\n\\\\".join(x.to_latex(show_k = show_k) for x in self.reactions)
        latex = f"\\left\\{{\\begin{{array}}{{l}}\n{latex}\n\\end{{array}}\\right."

        return latex
        
    @property
    def substances(self):
        pass 
    
    @property
    def reagents(self):
        pass

    @property
    def intermediates(self):
        pass
       
    
    @property
    def products(self):
        pass
    
        

In [35]:
reactions = [
    "A + B -> C",
    "A + B -> C",
    "A + B <- C",
    "2 A + 4 B -> 5 C",
    "4A + 5B -> 6C",
    "5 A + 6B -> 7 C",
    "2 H2O2 + 4D3 <-> 543 P3"
]

reac_chain = ReactionChain(reactions)

display(Latex(f"$${reac_chain.to_latex()}$$"))


 A +  B ->  C --True--  A +  B ->  C
 C <-  A +  B --True--  A +  B ->  C
 C <-  A +  B --True--  A +  B ->  C
2 A + 4 B -> 5 C --False--  A +  B ->  C
2 A + 4 B -> 5 C --False--  A +  B ->  C
2 A + 4 B -> 5 C --False--  C <-  A +  B
4 A + 5 B -> 6 C --False--  A +  B ->  C
4 A + 5 B -> 6 C --False--  A +  B ->  C
4 A + 5 B -> 6 C --False--  C <-  A +  B
4 A + 5 B -> 6 C --False-- 2 A + 4 B -> 5 C
5 A + 6 B -> 7 C --False--  A +  B ->  C
5 A + 6 B -> 7 C --False--  A +  B ->  C
5 A + 6 B -> 7 C --False--  C <-  A +  B
5 A + 6 B -> 7 C --False-- 2 A + 4 B -> 5 C
5 A + 6 B -> 7 C --False-- 4 A + 5 B -> 6 C
2 H2O2 + 4 D3 <-> 543 P3 --False--  A +  B ->  C
2 H2O2 + 4 D3 <-> 543 P3 --False--  A +  B ->  C
2 H2O2 + 4 D3 <-> 543 P3 --False--  C <-  A +  B
2 H2O2 + 4 D3 <-> 543 P3 --False-- 2 A + 4 B -> 5 C
2 H2O2 + 4 D3 <-> 543 P3 --False-- 4 A + 5 B -> 6 C
2 H2O2 + 4 D3 <-> 543 P3 --False-- 5 A + 6 B -> 7 C


<IPython.core.display.Latex object>

In [6]:
display(sym.Symbol("k_{-1}"))

k_{-1}